# AKANE E10-2：辞書の説明文・一般的な文を見せてからGemmaに聞く

このノートは **材料を取るだけ** です（比べは結果のファイルを読んで別に行います）。計画：sims/e10/PLAN_E10-2.md。

- 問題：E10-1 の問題（26,316 主張、主）と E10 の問題（47,170 主張、副）。正解はこのノートに入っていません。
- ⑩ 辞書（Wiktionary）の説明文を見せて聞く／⑩s 関係のない語の説明文を見せる（対照、E10-1 の問題だけ）
- ⑪g 一般的な文の資料（GenericsKB）から両方の語を含む文を最大5つ見せて聞く／⑪s 別の主張の文を見せる（対照、E10-1 の問題だけ）
- ⑪n 同じ文をNLIにかけて、主張が成り立つか
- 手で書いた語の一覧は使いません（語の形は lemminflect のデータで作る）。

## 手順
1. 「ランタイム」→「ランタイムのタイプを変更」→ **L4 GPU**。
2. 鍵マークの `HF_TOKEN` にアクセスを許可（E8〜E10-1と同じ）。
3. 「ランタイム」→「すべてのセルを実行」。ドライブの許可を聞かれたら許可。
4. 終わると、ドライブの `AKANE_アーキテクチャ/E10-2_results/` に `E102_*.json.gz` ができます。最後のセルが一覧と大きさを出します。
5. 時間の見込み：一般的な文の資料を取って索引を作る 10〜20分、Gemma 約20万回 60〜90分、NLI 10〜20分。合わせて1.5〜2時間。
6. 途中で切れても、もう一度「すべてのセルを実行」すれば、済んだ所は飛ばします。

In [ ]:
import os, json, time, gzip, subprocess, sys, hashlib, random
import numpy as np
from google.colab import drive, userdata
drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/AKANE_アーキテクチャ"
OUT = f"{BASE}/E10-2_results"
WIKT = f"{BASE}/E10-1_results/E101_wiktionary.json.gz"   # E10-1 で読んだ辞書
os.makedirs(OUT, exist_ok=True)
NLI_ID = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"   # E6〜E10-1と同じ
GEMMA_ID = "google/gemma-2-2b-it"
GKB_ID = "community-datasets/generics_kb"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lemminflect", "datasets"], check=False)
def save_gz(obj, name):
    p = f"{OUT}/{name}.json.gz"
    with gzip.open(p, "wt", encoding="utf-8") as f: json.dump(obj, f, ensure_ascii=False)
    print("保存:", name, round(os.path.getsize(p) / 1e6, 2), "MB"); return p
def load_gz(name):
    p = f"{OUT}/{name}.json.gz"
    if not os.path.exists(p): return None
    with gzip.open(p, "rt", encoding="utf-8") as f: return json.load(f)

In [ ]:
# 中身（sims/e10/e102/e102_core.py と同じ。手書きの語の一覧はありません）
"""E10-2の中身（Colabと手元の両方で使う）。計画：sims/e10/PLAN_E10-2.md。手で書いた語の一覧は使わない。
⑩ 辞書の説明文を見せてGemmaに聞く／⑩s 関係のない説明文を見せる（対照）
⑪g 一般的な文の資料（GenericsKB）から、物の語と分類の語の両方を含む文を取り、Gemmaに見せて聞く／⑪s 別の主張の文を見せる（対照）
⑪n 同じ文をNLIの前提にして、主張の文が成り立つか（辞書1段の③と同じ形）
語の形（複数形など）は lemminflect（辞書のデータ）で作る。手で書いた語形の一覧は使わない。
"""
import re

K_DEF, K_FACT, MAX_DEF_CHARS = 3, 5, 220
TOKEN = re.compile(r"[a-z0-9][a-z0-9\-']*")


def lookup(WKD, w):
    """E10-1 と同じ探し方：そのまま → 先頭の大文字/小文字を変えて → 'the ' を外して。"""
    for v in (w, w[:1].upper() + w[1:], w[:1].lower() + w[1:], re.sub(r"^[Tt]he ", "", w)):
        if v in WKD:
            return v
    return None


def defs_text(WKD, w, k=K_DEF):
    """ページに出る順に最初の k 個の意味（印による選び分けはしない）。"""
    key = lookup(WKD, w)
    if not key:
        return "(not in the dictionary)"
    ss = [s[0] if isinstance(s, (list, tuple)) else s["def"] for s in WKD[key][:k]]
    return "; ".join(d[:MAX_DEF_CHARS] for d in ss) if ss else "(not in the dictionary)"


def class_term(dom, c, meta):
    """分類の側の語（辞書を引く語・文を探す語）。P は全体の語、D は分類名、G は地域名。"""
    if dom == "P":
        return c
    if dom == "D":
        return meta["labels"][c]["name"]
    return meta["regions"][c]


def def_prompt(x, xd, w, wd, stmt):
    return (f"Dictionary definitions:\n\"{x}\": {xd}\n\"{w}\": {wd}\n\n"
            f"Using these definitions and your own knowledge, is the following statement true? Answer Yes or No only.\n"
            f"Statement: {stmt}")


def fact_prompt(facts, stmt):
    body = "\n".join(f"- {f}" for f in facts) if facts else "(no facts found)"
    return (f"Facts:\n{body}\n\n"
            f"Using these facts and your own knowledge, is the following statement true? Answer Yes or No only.\n"
            f"Statement: {stmt}")


def gemma_q(stmt):
    return f"Is the following statement true? Answer Yes or No only.\nStatement: {stmt}"   # E4〜E10-1と同じ


def tokens(s):
    return TOKEN.findall(s.lower())


def forms(term, infl):
    """語（または語のまとまり）の形の集まり：そのままの形と、最後の語の名詞の変化形（infl＝lemminflect.getAllInflections）。"""
    ws = tokens(term)
    if not ws:
        return set()
    out = {tuple(ws)}
    for vs in infl(ws[-1], upos="NOUN").values():
        for v in vs:
            out.add(tuple(ws[:-1] + [v.lower()]))
    return out


def find_spans(toks, form):
    n = len(form)
    return any(tuple(toks[i:i + n]) == form for i in range(len(toks) - n + 1))


def term_hits(sents, scores, term_forms, log=print, every=500000):
    """一般的な文を score の高い順に並べ、各語（の形のどれか）を含む文の番号の集まりを返す。
    term_forms: {語: {形(語の並び), ...}}。文は一度だけ読む。返す番号は並べ替えた後の番号（小さいほど score が高い）。"""
    order = sorted(range(len(sents)), key=lambda i: -scores[i])
    by_first = {}
    for t, fs in term_forms.items():
        for f in fs:
            by_first.setdefault(f[0], []).append((t, f))
    hits = {t: set() for t in term_forms}
    kept = {}
    for r, i in enumerate(order):
        toks = tokens(sents[i])
        found = False
        for w in set(toks) & by_first.keys():
            for t, f in by_first[w]:
                if len(f) == 1 or find_spans(toks, f):
                    hits[t].add(r); found = True
        if found:
            kept[r] = sents[i]
        if every and r % every == 0:
            log(f"  文 {r:,}/{len(order):,}")
    return hits, kept

In [ ]:
SETS = json.loads(r'''{"N": {"meta": {"wholes": {"architecture": "an architecture", "bacteria": "a bacteria", "beef": "a beef", "bridge": "a bridge", "bud": "a bud", "cat": "a cat", "chapel": "a chapel", "chick": "a chick", "clock": "a clock", "deer": "a deer", "dog": "a dog", "elephant": "an elephant", "embassy": "an embassy", "garage": "a garage", "highway": "a highway", "kidney": "a kidney", "lamb": "a lamb", "male": "a male", "marijuana": "a marijuana", "puppy": "a puppy", "receiver": "a receiver", "reed": "a reed", "rifle": "a rifle", "road": "a road", "salmon": "a salmon", "shark": "a shark", "sheep": "a sheep", "shook": "a shook", "tank": "a tank", "wheat": "a wheat"}, "labels": {"cardio": {"name": "cardiovascular system disease", "art": "a"}, "resp": {"name": "respiratory system disease", "art": "a"}, "gi": {"name": "gastrointestinal system disease", "art": "a"}, "nerv": {"name": "nervous system disease", "art": "a"}, "skin": {"name": "skin disease", "art": "a"}, "musc": {"name": "musculoskeletal system disease", "art": "a"}, "urin": {"name": "urinary system disease", "art": "a"}, "endo": {"name": "endocrine system disease", "art": "an"}, "immune": {"name": "immune system disease", "art": "an"}, "blood": {"name": "blood disease", "art": "a"}, "repro": {"name": "reproductive system disease", "art": "a"}, "breast": {"name": "breast disease", "art": "a"}, "chest": {"name": "thoracic disease", "art": "a"}, "metab": {"name": "metabolic disease", "art": "a"}, "infect": {"name": "infectious disease", "art": "an"}, "tumor": {"name": "cancer or other tumor", "art": "a"}, "mental": {"name": "mental disorder", "art": "a"}, "genetic": {"name": "genetic disease", "art": "a"}, "bact": {"name": "bacterial infection", "art": "a"}, "viral": {"name": "viral infection", "art": "a"}, "fungal": {"name": "fungal infection", "art": "a"}, "parasit": {"name": "parasitic infection", "art": "a"}}, "regions": {"Central America": "Central America", "Western Asia": "Western Asia", "Eastern Asia": "Eastern Asia", "Western Europe": "Western Europe", "South-eastern Asia": "South-Eastern Asia", "Northern Africa": "Northern Africa", "Australia and New Zealand": "Australia and New Zealand", "Melanesia": "Melanesia", "Western Africa": "Western Africa", "Polynesia": "Polynesia", "Southern Africa": "Southern Africa", "Southern Europe": "Southern Europe", "Northern Europe": "Northern Europe", "Caribbean": "Caribbean", "Central Asia": "Central Asia", "Micronesia": "Micronesia", "Southern Asia": "Southern Asia", "Middle Africa": "Middle Africa", "Northern America": "Northern America", "Eastern Africa": "Eastern Africa", "Eastern Europe": "Eastern Europe", "South America": "South America"}}, "order": {"P": ["action", "aerial", "annex", "antenna", "anterior", "antique", "apartment", "aperture", "approach", "arch", "ash", "ash-key", "axle", "balance", "bang", "barb", "barrel", "base", "batting", "beak", "beard", "beetle", "bell", "belly", "bend", "bill", "blood", "blue-belly", "boiler", "bolt", "bottle", "bottle-tree", "boulevard", "brain", "brain-stem", "brake", "breast", "brim", "brow", "bulb", "bullet", "burger", "bypass", "canal", "cannabis", "cannon", "cap", "capacitor", "capital", "cardinal", "carpet", "carving", "case", "casing", "cathode", "ceiling", "chart", "cheat", "cheek", "chest", "chin", "choke", "chromosome", "circle", "circuit", "cleaner", "clip", "clone", "club", "coat", "cock", "comb", "cone", "contact", "cope", "coping", "corner", "cornerstone", "corolla", "corona", "corpus", "cortex", "costa", "course", "court", "courtyard", "cradle", "crest", "crossing", "crown", "curve", "dashboard", "dial", "dice", "differential", "digit", "dock", "door", "doorstep", "doorway", "driver", "duct", "ear", "edible", "electrode", "elevator", "extension", "eyeball", "eyebrow", "eyelid", "facial", "factor", "feather", "feature", "fence", "fin", "finder", "flag", "floor", "flooring", "foil", "food", "forehead", "foundation", "foyer", "frame", "funeral-residence", "fungus", "gene", "god", "goose", "graphic", "grid", "groundwork", "gum", "gun-sight", "hair", "half-moon", "hall", "hammer", "header", "heading", "heat", "heating", "hemisphere", "hen", "hind", "hinge", "hood", "hoof", "hoop", "horse-trail", "interchange", "intersection", "iris", "ivy", "jack", "jacket", "jaw", "jelly", "jersey", "jewel", "jimmy", "john", "key", "keystone", "lag", "landing", "lash", "left", "lens", "level", "lid", "lift", "lighthouse", "lip", "lobby", "lobe", "lock", "lounge", "magazine", "mast", "matrix", "mock-up", "monastery", "moss", "moustache", "movement", "mustache", "muzzle", "nail", "nasal", "nib", "nook", "nose", "nucleus", "oculus", "optic", "orb", "orbit", "pad", "palate", "pale", "pane", "panther", "part", "paw", "peak", "pearl-fish", "pelvis", "petal", "photograph", "pier", "pineapple", "placenta", "plain", "plate", "plume", "poison-berry", "portion", "portrait", "product", "punch", "pupil", "quill", "radiator", "ray", "relay", "resistance", "resistor", "retina", "rib", "rim", "ring", "roadside", "rod", "roe", "roof", "room", "rotary", "roundabout", "row", "ruin", "rump", "saddle", "saliva", "sash", "scum", "section", "segment", "serum", "shaft", "shank", "sheep-tick", "shift", "shoot", "short", "shoulder", "sights", "skeleton", "skull", "slide", "smoke", "snout", "soldier-fish", "spit", "springer", "stair", "staircase", "stairway", "stave", "step", "stigma", "stock", "storey", "story", "stretcher", "stud", "style", "sucker", "sushi", "tail", "tandem", "tap", "target", "teeth", "temple", "thick-knee", "threshold", "time-ball", "toe", "tom-tom", "tongue", "tongue-flower", "tornado", "total", "track", "tread", "trigger", "trunk", "tube", "turnaround", "turnout", "turret", "tyre", "upstairs", "vaccine", "vane", "variation", "vas", "vent-hole", "vertical", "vessel", "volleyball", "wall", "wave", "web", "weed", "weeds", "wheel", "widening", "window", "wing", "wire", "work-clothing"], "D": ["acute diarrhea", "aspiration pneumonia", "swine influenza", "Stevens-Johnson syndrome", "adult T-cell leukemia/lymphoma", "abdominal tuberculosis", "alcohol-related birth defects", "organ system cancer", "Dent disease", "vitamin metabolic disorder", "nicotine dependence", "Ogden syndrome", "sensory system cancer", "heavy chain disease", "delta chain disease", "acquired metabolic disease", "Kennedy's disease", "juvenile absence epilepsy", "stuttering", "peeling skin syndrome", "gastrointestinal allergy", "apple allergy", "plum allergy", "carp allergy", "white shrimp allergy", "snail allergy", "cardiac tuberculosis", "congenital diarrhea", "early-onset Parkinson's disease", "vegetable allergy", "mitochondrial Parkinson's disease", "vascular-type bleeding disorder", "bone remodeling disease", "Coronavirus infectious disease", "kidney cortex disease", "chronic asthma", "acute asthma", "nocturnal asthma", "subjective cognitive decline", "parsley allergy", "traumatic brain injury", "Fraser syndrome", "enhanced S-cone syndrome", "Ballard syndrome", "SHORT syndrome", "Gordon Holmes syndrome", "CHILD syndrome", "CK syndrome", "pelvic inflammatory disease", "small intestine cancer", "breast cyst", "stomach cancer", "intellectual disability", "short bowel syndrome", "separation anxiety disorder", "renal hypertension", "kidney failure", "premature menopause", "malignant hypertension", "tooth disease", "borderline personality disorder", "dependent personality disorder", "obsessive-compulsive disorder", "multiple personality disorder", "impulse control disorder", "lens disease", "trench fever", "neonatal respiratory failure", "T cell deficiency", "acute orbital inflammation", "appendix cancer", "appendix lymphoma", "rabies", "expressive language disorder", "acquired night blindness", "Chandler syndrome", "functional gastric disease", "pelvic muscle wasting", "allergic disease", "cartilage disease", "globe disease", "hepatitis A", "diarrhea", "reading disorder", "Stiff-Person syndrome", "color blindness", "secondary Parkinson disease", "nutmeg liver", "excessive tearing", "red-green color blindness", "cerebral malaria", "Masters-Allen syndrome", "frozen shoulder", "Machado-Joseph disease", "cerebral degeneration", "geographic tongue", "primary pulmonary hypertension", "FG syndrome", "Larsen syndrome", "colon carcinoma", "colon lymphoma", "alcohol use disorder", "acute chest syndrome", "childhood absence epilepsy", "epilepsy", "cannabis dependence", "skull cancer", "optic nerve disease", "placenta cancer", "communication disorder", "hepatitis D", "autoimmune hepatitis", "factor XIII deficiency", "factor XII deficiency", "hepatitis", "cervix disease", "lateral sclerosis", "West Nile fever", "multiple sclerosis", "iris disease", "tooth erosion", "alcoholic psychosis", "Gilbert syndrome", "auditory system disease", "narcissistic personality disorder", "long QT syndrome", "familial Mediterranean fever", "substance-related disorder", "spinal cord disease", "plague", "cerebral arterial disease", "anterior compartment syndrome", "heart valve disease", "bile duct disease", "skin cancer", "Alexander disease", "external ear carcinoma", "smooth muscle tumor", "retinal cancer", "Nelson syndrome", "thoracic cancer", "ear cancer", "inner ear cancer", "simple partial epilepsy", "infertility", "inhibited male orgasm", "external ear cancer", "pulmonary valve disease", "composite lymphoma", "adult lymphoma", "benign vascular tumor", "reactive arthritis", "acne", "heart lymphoma", "chest wall lymphoma", "lung lymphoma", "eye lymphoma", "mitochondrial metabolism disease", "B-cell lymphoma", "childhood leukemia", "placenta disease", "cocaine abuse", "cork-handlers' disease", "gum cancer", "upper lip cancer", "smallpox", "intermediate coronary syndrome", "Wilson disease", "Evans' syndrome", "lower gum cancer", "allergic asthma", "Sheehan syndrome", "blue drum syndrome", "peripheral vertigo", "congenital heart block", "lymph node disease", "cocaine dependence"], "G": ["Bani Yas City", "Al Karama", "The Palm Jumeirah", "Al Mizhar First", "Khulm", "Taloqan", "Eslam Qaleh", "Karukh", "Elbasan", "Berat", "Fier", "Peqin", "Luquembo", "N'zeto", "Quipungo", "Kunda dya Baze", "Lomas de Zamora", "Caleta Olivia", "Lobos", "La Falda", "Brigittenau", "Alsergrund", "Mariahilf", "Sankt Leonhard", "Caloundra", "Randwick", "Dandenong North", "Hawthorn", "Banja Luka", "Bugojno", "Doboj", "Cazin", "Savar", "Barishal", "Uttar Char Fasson", "Srimangal", "Forest", "Izegem", "Watermael-Boitsfort", "Oostkamp", "Koudougou", "Orodara", "Korsimoro", "Karankasso", "Stara Zagora", "Kazanlak", "Targovishte", "Dupnitsa", "Ngozi", "Rumonge", "Kayanza", "Bubanza", "Bassila", "Tchaourou", "Kpozoun", "Toukountouna", "Sengkurong", "Seria", "Tutong", "Liang", "Riberalta", "Bermejo", "San Ignacio de Velasco", "Tupiza", "Angra dos Reis", "Serrinha", "Nova Soure", "Campos Belos", "Tlokweng", "Kanye", "Thamaga", "Janeng", "Orsha", "Slonim", "Novogrudok", "Ivatsevichi", "Downtown Vancouver", "Courtenay", "Willowridge-Martingrove-Richview", "Clarence-Rockland", "Bukavu", "Mwene", "Djugu", "Kabare", "Bocaranga", "Batangafo", "Dekoa", "Kabo", "Owando", "Gamboma", "Mouyondzi", "Pokola", "Sankt Gallen", "Sitten", "Yverdon-les-Bains", "Wettingen", "Katiola", "Tanda", "Diourouzon", "Nzianouan", "La Serena", "Renca", "Hacienda La Calera", "Angol", "Kribi", "Tibati", "Figuil", "Makak", "Puqi", "Qiuji", "Shiqiaozi", "Chenlou", "Santa Marta", "San Gil", "Anaime", "Cota", "San Vicente de Moravia", "Quesada", "Esparza", "San Juan de Dios", "Moa", "Placetas", "Calimete", "La Sierpe", "Pilsen", "Opava", "Michle", "Bohnice", "Marzahn", "Bad Kreuznach", "Hille", "Vechelde", "Randers", "Hedensted", "Glostrup", "Middelfart", "Bonao", "Villa Francisca", "Salcedo", "Hato Mayor del Rey", "Sedrata", "Djidiouia", "Grarem", "Charef", "Ambato", "Machala", "Ibarra", "Rosa Zarate", "Tartu", "Haabersti", "Kristiine", "Viljandi", "Rivas-Vaciamadrid", "Chipiona", "Castilleja de la Cuesta", "Carlet", "Kibre Mengist", "Tulu Bolo", "Deder", "Adebaye", "East Helsinki", "Kuopio", "Herttoniemi", "Raahe", "Troyes", "Rosny-sous-Bois", "Savigny-le-Temple", "Le Pontet", "Oyem", "Bitam", "Oyam", "Makokou", "Bath", "Bicester", "Bideford", "Belle Vale", "Sekondi-Takoradi", "Kade", "Begoro", "Nalerigu", "Nema Kunku", "Lamin", "Kotu", "Tanji", "Camayenne", "Fria", "Mandiana", "Moussadou", "Le Gosier", "Petit-Bourg", "Le Moule", "Basse-Terre", "Volos", "Agios Dimitrios", "Mytilene", "Neapoli", "Zacapa", "Yepocapa", "Alotenango", "Tiquisate", "Bairro Militar", "Reino de Bandim", "Reino de Antula", "Plaque", "Whampoa", "Shui Chuen O", "Wo Che", "Mid Levels", "El Progreso", "La Lima", "Macuelizo", "Marcovia", "Karlovac", "Velika Gorica", "Bjelovar", "Liberpolis", "Croix-des-Bouquets", "Jacmel", "Verrettes", "Lenbe", "Nagykanizsa", "Baja", "Keszthely", "Dunaharaszti", "Letterkenny", "Greystones", "Balbriggan", "Lucan", "Ramat Gan", "Jaffa", "Givatayim", "Or Yehuda", "Pallichal", "Kotagiri", "Palavakkam", "Govardhan", "Basrah", "Najaf", "Al Diwaniyah", "Kufa", "Kermanshah", "Sirjan", "Hashtgerd", "Hoveyzeh", "Circoiscrizione IV", "Bacoli", "Villaricca", "Fossano", "Mandeville", "May Pen", "Ocho Rios", "Half Way Tree", "Zarqa", "Irbid", "Marj Al Hamam", "Judita", "Tochigi", "Maruobara", "Yoshioka", "Makubetsu", "Mombasa", "Eldoret", "Garissa", "Kimilili", "Balykchy", "Razzakov", "Iradan", "Tash-Kumyr", "Kampong Cham", "Pursat", "Lumphat", "Banlung", "Rajin", "Hoeyang", "Chunghwa", "Samho-rodongjagu", "Hwaseong-si", "Masan", "Gwangyang", "Tallang-dong", "Ban Khoan", "Xam Nua", "Pakxan", "Sekong", "Aley", "Jounieh", "Ghazieh", "El Hermel", "Jaffna", "Ambalangoda", "Kandana", "Hambantota", "Buchanan", "Ganta", "Harbel", "Saclepea", "Maputsoe", "Mohale's Hoek", "Mafeteng", "Thaba-Tseka", "Kaunas", "Eiguliai", "Naujininkai", "Telsiai", "Daugavpils", "Ventspils", "Valmiera", "Tukums", "Sirte", "Zuwarah", "Ubari", "Brak", "Skhirate", "Tinghir", "Ait Ourir", "Kelaat Mgouna", "Tiraspol", "Comrat", "Drochia", "Codru", "Ambanja", "Hell-Ville", "Didy", "Amboanjo", "Tetovo", "Kochani", "Lipkovo", "Zelino", "Douentza", "Finnkolo", "Kouri", "Goundam", "Kyimyindine", "Aungban", "Zee Kone", "Yesagyo", "Nalayh", "Khovd", "Tsetserleg", "Dzuunmod", "Mbera", "Tevragh Zeina", "Aleg", "Tembedgha", "Beau Bassin-Rose Hill", "Quatre Bornes", "Goodlands", "Baie du Tombeau", "Salima", "Liwonde", "Ntcheu", "Monkey Bay", "Agua Prieta", "La Barca", "Zacoalco de Torres", "Libres", "Hulu Langat", "Dungun", "Sungkai", "Kampung Tanjung Karang", "Monapo", "Gondola", "Namacurra", "Pebane", "Rundu", "Swakopmund", "Oshakati", "Ondangwa", "Agadez", "Nguigmi", "Mayahi", "Torodi", "Abeokuta", "Agege", "Nguru", "Ipoti", "Chinandega", "Matagalpa", "Mateare", "Juigalpa", "Nieuwegein", "Hoogeveen", "Ommoord", "Zeewolde", "Nesttun", "Halden", "Harstad", "Kongsberg", "Madhyapur Thimi", "Panauti", "Siraha", "Dailekh", "Porirua", "Whangarei", "Paraparaumu", "Karori", "Rustaq", "Sohar", "Sur", "Adam", "La Chorrera", "Chilibre", "Changuinola", "Sabanitas", "Talara", "Abancay", "Picsi", "Quillabamba", "Brookes Point", "Itogon", "Jaen", "Pililla", "Dera Ismail Khan", "Phalia", "Sillanwali", "Harappa", "Konin", "Nysa", "Krapkowice", "Osowa", "Caguas", "Trujillo Alto", "Barceloneta", "Aguadilla", "Hebron", "Old City", "Surif City", "Jericho", "Vila Nova de Gaia", "Arrentela", "Pontinha", "Baguim do Monte", "Fernando de la Mora", "Villa Hayes", "Coronel Oviedo", "Loma Plata", "Turda", "Lugoj", "Gherla", "Gura Humorului", "Novi Pazar", "Kikinda", "Sremska Mitrovica", "Vrbas", "Cyangugu", "Nyabitekeli", "Kayonza", "Umutara", "Madinah", "Sakakah", "King Faisal Military City", "Afif", "Kassala", "El Geneina Fort", "Kadugli", "Al Hasaheisa", "Vasastaden", "Tumba", "Sundbyberg", "Gustavsberg", "Kangkar", "Senja", "Telok Blangah", "Tiong Bahru Estate", "Maribor", "Kranj", "Novo Mesto", "Trbovlje", "Trnava", "Michalovce", "Bardejov", "Malacky", "Kenema", "Port Loko", "Lunsar", "Goderich", "Kolda", "Mboro", "Kayar", "Kbombole", "Berbera", "Balanbale", "Dhabad", "Ceerigaabo", "Pontbuiten", "Rainville", "Lelydorp", "Tammenga", "Malakal", "Wau", "Aweil", "Kafia Kingi", "Santa Tecla", "Delgado", "San Rafael Oriente", "Sonzacate", "Homs", "Al-Tabqa", "Al Qaryatayn", "Tallkalakh", "Kelo", "Doba", "Pala", "Massaguet", "Bassar", "Sotouboua", "Bafilo", "Tabligbo", "Suan Luang", "Bangkok Yai", "Pran Buri", "Bang Bon Nuea", "Boshkengash", "Norak", "Istiqlol", "Vakhsh", "Balkanabat", "Bayramaly", "Yylanly", "Gubadag", "Menzel Bourguiba", "El Kef", "Jendouba", "Borj Cedria", "Arima", "Diego Martin", "Point Fortin", "Tunapuna", "Mbuguni", "Namanyere", "Ngudu", "Kibara", "Kryukiv", "Debaltseve", "Korostyshiv", "Bakhmach", "Butaleja", "Anaka", "Lwakyomba", "Iryango", "The Villages", "Southglenn", "Wheat Ridge", "Carteret", "Fray Bentos", "Treinta y Tres", "Carmelo", "Young", "Kogon Shahri", "Kitob", "Shofirkon Shahri", "Chelak", "Ciudad Guayana", "La Victoria", "Carora", "La Azulita", "Haiphong", "Phu Quoc", "Lai Vung", "Mui Ne", "Al Bayda", "Ataq", "Ma'rib", "Mocha", "Centurion", "Kroonstad", "Harrismith", "Burgersdorp", "Mufulira", "Luwingu", "Manyinga", "Chiengi", "Chegutu", "Chiredzi", "Gokwe", "Gwanda"]}, "corder": {"P": ["architecture", "bacteria", "beef", "bridge", "bud", "cat", "chapel", "chick", "clock", "deer", "dog", "elephant", "embassy", "garage", "highway", "kidney", "lamb", "male", "marijuana", "puppy", "receiver", "reed", "rifle", "road", "salmon", "shark", "sheep", "shook", "tank", "wheat"], "D": ["cardio", "resp", "gi", "nerv", "skin", "musc", "urin", "endo", "immune", "blood", "repro", "breast", "chest", "metab", "infect", "tumor", "mental", "genetic", "bact", "viral", "fungal", "parasit"], "G": ["Australia and New Zealand", "Caribbean", "Central America", "Central Asia", "Eastern Africa", "Eastern Asia", "Eastern Europe", "Melanesia", "Micronesia", "Middle Africa", "Northern Africa", "Northern America", "Northern Europe", "Polynesia", "South America", "South-eastern Asia", "Southern Africa", "Southern Asia", "Southern Europe", "Western Africa", "Western Asia", "Western Europe"]}, "md5": "bb5538d2f0", "n": 26316}, "O": {"meta": {"wholes": {"body": "a human body", "head": "a head", "face": "a face", "hand": "a hand", "foot": "a foot", "eye": "an eye", "mouth": "a mouth", "arm": "an arm", "leg": "a leg", "tooth": "a tooth", "bird": "a bird", "fish": "a fish", "horse": "a horse", "tree": "a tree", "flower": "a flower", "car": "a car", "bicycle": "a bicycle", "airplane": "an airplane", "ship": "a ship", "boat": "a boat", "house": "a house", "church": "a church", "computer": "a computer", "watch": "a watch", "piano": "a piano", "gun": "a gun", "shoe": "a shoe", "shirt": "a shirt", "camera": "a camera", "knife": "a knife"}, "labels": {"cardio": {"name": "cardiovascular system disease", "art": "a", "system": "cardiovascular system"}, "resp": {"name": "respiratory system disease", "art": "a", "system": "respiratory system"}, "gi": {"name": "gastrointestinal system disease", "art": "a", "system": "digestive system"}, "nerv": {"name": "nervous system disease", "art": "a", "system": "nervous system"}, "skin": {"name": "skin disease", "art": "a", "system": "skin"}, "musc": {"name": "musculoskeletal system disease", "art": "a", "system": "musculoskeletal system"}, "urin": {"name": "urinary system disease", "art": "a", "system": "urinary system"}, "endo": {"name": "endocrine system disease", "art": "an", "system": "endocrine system"}, "immune": {"name": "immune system disease", "art": "an", "system": "immune system"}, "blood": {"name": "blood disease", "art": "a", "system": "blood"}, "repro": {"name": "reproductive system disease", "art": "a", "system": "reproductive system"}, "breast": {"name": "breast disease", "art": "a", "system": "breast"}, "chest": {"name": "thoracic disease", "art": "a", "system": "chest"}, "metab": {"name": "metabolic disease", "art": "a", "system": null}, "infect": {"name": "infectious disease", "art": "an", "system": null}, "tumor": {"name": "cancer or other tumor", "art": "a", "system": null}, "mental": {"name": "mental disorder", "art": "a", "system": null}, "genetic": {"name": "genetic disease", "art": "a", "system": null}, "bact": {"name": "bacterial infection", "art": "a", "system": null}, "viral": {"name": "viral infection", "art": "a", "system": null}, "fungal": {"name": "fungal infection", "art": "a", "system": null}, "parasit": {"name": "parasitic infection", "art": "a", "system": null}}, "regions": {"Micronesia": "Micronesia", "Eastern Europe": "Eastern Europe", "Western Europe": "Western Europe", "Melanesia": "Melanesia", "Middle Africa": "Middle Africa", "Eastern Asia": "Eastern Asia", "Northern America": "Northern America", "Southern Africa": "Southern Africa", "Western Asia": "Western Asia", "South-eastern Asia": "South-Eastern Asia", "Southern Asia": "Southern Asia", "Western Africa": "Western Africa", "Central Asia": "Central Asia", "South America": "South America", "Caribbean": "Caribbean", "Central America": "Central America", "Eastern Africa": "Eastern Africa", "Northern Europe": "Northern Europe", "Southern Europe": "Southern Europe", "Northern Africa": "Northern Africa", "Polynesia": "Polynesia", "Australia and New Zealand": "Australia and New Zealand"}}, "order": {"P": ["ab", "abdomen", "abdominal", "accelerator", "action", "air-intake", "airway", "anchor", "ankle", "annex", "anus", "aperture", "appendix", "arch", "argentine", "armpit", "arse", "articulation", "ass", "attic", "axle", "back", "backside", "balance", "bale", "ball", "band", "barb", "bark", "barker", "barrel", "bars", "base", "basic", "bath", "bathroom", "bathtub", "baton", "bay", "beak", "beard", "bearing", "bed", "bedroom", "behind", "belly", "belt", "bill", "bishop", "blade", "block", "bloodstream", "bloom", "blossom", "boiler", "bolt", "bomber", "bonnet", "bosom", "bottom", "bow", "bowel", "brain", "brain-stem", "brake", "brakes", "breast", "bridge", "brow", "brush", "buffer", "buffet", "bulb", "bum", "bumper", "buns", "bus", "butt", "butter", "buttocks", "cab", "cabin", "calf", "can", "cannon", "canopy", "cap", "capacitor", "capsule", "cartridge", "case", "casing", "cavity", "ceiling", "cement", "cervix", "chain", "chamber", "charcoal", "chassis", "cheek", "cherry", "chest", "chin", "chip", "choir", "choke", "chromosome", "clarence", "clip", "closure", "clothing", "clutch", "coat", "cock", "cockpit", "coffin", "collar", "compartment", "condo", "cone", "conjunction", "control", "cope", "coping", "corner", "cornerstone", "corolla", "corona", "cortex", "costa", "counter", "course", "court", "courtyard", "cross", "crotch", "crown", "crystal", "cuff", "cushion", "cusp", "cylinder", "dart", "dashboard", "deck", "den", "dial", "diaphragm", "digit", "dining-room", "disposable", "distributor", "dock", "door", "doorstep", "doorway", "drawer", "drawing", "duct", "ear", "earth", "elbow", "elevated", "elevator", "emergency", "enamel", "endocrine", "escape", "exhaust", "extension", "extremity", "eyeball", "eyebrow", "eyelid", "facial", "facing", "fall-board", "fan", "fanny", "feather", "feature", "femur", "fender", "fetus", "film", "fin", "finder", "finger", "firearm", "first", "flap", "flaps", "flood", "floor", "flooring", "floppy", "fluke", "fly", "fore", "forearm", "foreground", "forehead", "forget-me-not", "fork", "foundation", "foyer", "fraction", "frame", "funnel", "fuselage", "galley", "gap", "gas", "gather", "gathering", "gay-feather", "gear", "gearbox", "gene", "gorge", "graphics", "grille", "grip", "groin", "groundwork", "guitar", "gum", "gun-sight", "gut", "hair", "half-moon", "hall", "hammer", "handle", "hank", "harbour", "hardware", "header", "heart", "heat", "heating", "heel", "helm", "hemisphere", "high", "hinge", "hip", "hold", "hood", "hoof", "horn", "horseback", "hub", "hull", "ignition", "insulin", "intestine", "iris", "item", "jade", "jag", "jaw", "joint", "journal", "keel", "key", "keyboard", "kitchen", "knee", "knee-hi", "knuckle", "lace", "landing", "lap", "lash", "lat", "leaflet", "lens", "level", "library", "lid", "lien", "lift", "ligament", "limb", "liner", "lining", "lip", "litter-basket", "liver", "living-room", "load", "lobby", "lobe", "lock", "loft", "log", "low", "lure", "lymph", "magazine", "magneto", "mallet", "marina", "marrow", "matrix", "mattress", "memory", "middle", "milk", "missile", "mitt", "mixer", "monitor", "mortar", "mosque", "moth", "moustache", "movement", "muscle", "mustache", "muzzle", "nail", "nasal", "nave", "neck", "negative", "network", "neuron", "nib", "node", "nook", "nose", "nucleus", "oculus", "optic", "orb", "orbit", "organ", "os", "outfit", "pad", "palate", "palm", "pancreas", "pane", "parachute", "parlor", "parlour", "pavilion", "paw", "pawn", "pedal", "peel", "peg", "pen-and-ink", "peripheral", "pes", "petal", "picnic", "pier", "pixel", "placenta", "plate", "plug", "plume", "pocket", "pod", "point", "poll", "pool", "poop", "porch", "portfolio", "posterior", "pot", "potty", "processor", "produce", "pulp", "pump", "pupil", "quad", "quarry", "quarter", "quill", "radiator", "radius", "ray", "ready-mix", "ready-to-wear", "rear", "receptor", "register", "retina", "retreat", "reverse", "rib", "ride", "rim", "rod", "roe", "roof", "room", "root", "rose", "rotor", "row", "rudder", "rug", "rump", "saddle", "saliva", "sanctuary", "sash", "scooter", "scope", "screen", "screw", "seal", "seat", "seatbelt", "second", "self-starter", "set", "shaft", "shank", "she-goat", "sheet", "shelf", "shifter", "shin", "shock", "shoulder", "shower", "shroud", "shutter", "side", "six-gun", "skeleton", "skin", "skirt", "skull", "sleeve", "small", "snail", "snout", "socket", "sole", "soundtrack", "soy", "spar", "specific", "spike", "spit", "splash-guard", "spleen", "spoke", "squash", "stair", "staircase", "stairway", "starter", "stem", "step", "stern", "sticker", "stigma", "stock", "stomach", "stool", "stop", "storage", "store", "storey", "story", "stretcher", "string", "stripe", "stud", "study", "stump", "style", "sugar-bush", "sunshine-roof", "suspension", "swan", "tack", "tail", "tailor-made", "teeth", "temple", "tendon", "thermostat", "thigh", "third", "threshold", "throat", "throne", "throttle", "ticker", "time-ball", "toe", "toilet", "tongue", "top", "topping", "torso", "tract", "transmission", "tread", "trigger", "trunk", "tub", "tum", "tummy", "two", "universal", "upper", "upstairs", "valve", "vane", "vas", "vein", "vessel", "waist", "wall", "wardrobe", "ware", "web", "wheel", "wheeler", "wicket", "winch", "window", "windshield", "wing", "work-clothes", "wrapping", "wrist", "yoke"], "D": ["shrimp allergy", "aspirin allergy", "nickel allergic asthma", "cobalt allergic asthma", "mineral metabolism disease", "African tick-bite fever", "Japanese spotted fever", "inflammatory diarrhea", "Pontiac fever", "sensory system disease", "restless legs syndrome", "Usher syndrome", "yellow nail syndrome", "Costello syndrome", "Carney complex", "spinal polio", "Ross River fever", "West syndrome", "inflammatory bowel disease", "ADULT syndrome", "triple-A syndrome", "respiratory system cancer", "Rh deficiency syndrome", "Arts syndrome", "fetal alcohol syndrome", "female breast cancer", "mixed cerebral palsy", "small cell carcinoma", "cell type cancer", "anal canal cancer", "alcohol dependence", "mantle cell lymphoma", "marginal zone lymphoma", "peripheral T-cell lymphoma", "ARC syndrome", "short QT syndrome", "artery disease", "peripheral artery disease", "sleep apnea", "renal fibrosis", "ovarian melanoma", "stroke", "withdrawal disorder", "Good syndrome", "dropped head syndrome", "specific developmental disorder", "pervasive developmental disorder", "autism spectrum disorder", "atypical autism", "sexual health disorder", "writing disorder", "gluten allergy", "lymphoma", "Pearson syndrome", "triple-negative breast cancer", "immune system cancer", "cartilage cancer", "alpha chain disease", "mu chain disease", "pain disorder", "bipolar ll disorder", "Timothy syndrome", "Carpenter syndrome", "UV-sensitive syndrome", "specific language impairment", "Mast syndrome", "cold-induced sweating syndrome", "cardiac arrest", "Cole-Carpenter syndrome", "EAST syndrome", "Perry syndrome", "Pitt-Hopkins syndrome", "chicken egg allergy", "shellfish allergy", "respiratory allergy", "pollen allergy", "Timothy grass allergy", "drug allergy", "metal allergy", "fruit allergy", "cherry allergy", "Indian plum allergy", "orange allergy", "melon allergy", "peach allergy", "tomato allergy", "fish allergy", "Atlantic cod allergy", "Atlantic salmon allergy", "rainbow trout allergy", "cow milk allergy", "goat milk allergy", "brown shrimp allergy", "crab allergy", "latex allergy", "Char syndrome", "spinal disease", "abdominal obesity-metabolic syndrome", "Peters anomaly", "hand-foot-genital syndrome", "Wilson-Turner syndrome", "juvenile-onset Parkinson's disease", "Norman-Roberts syndrome", "White-Sutton syndrome", "absence epilepsy", "drug-induced hearing loss", "celery allergy", "Watson syndrome", "Mitchell syndrome", "ring chromosome syndrome", "Ramon syndrome", "alcohol-associated liver disease", "alpha-gal syndrome", "Davis-Wells syndrome", "HELIX syndrome", "lung-brain developmental disorder", "muscular disease", "bone disease", "bone development disease", "bone deterioration disease", "X-linked dominant disease", "bone structure disease", "Worth syndrome", "intestinal pseudo-obstruction", "PAPA syndrome", "Sweeney-Cox syndrome", "tongue carcinoma", "NK cell deficiency", "gene duplication disease", "Sweet syndrome", "urinary tract infection", "intermittent asthma", "persistent mild asthma", "persistent moderate asthma", "childhood-onset asthma", "adult-onset severe asthma", "environmentally induced asthma", "occupational asthma", "aspirin-induced respiratory disease", "near-fatal asthma", "persistent severe asthma", "thunderstorm triggered asthma", "mild cognitive impairment", "long COVID", "hot water epilepsy", "injection anthrax", "graft-versus-host disease", "breast implant illness", "Becker disease", "semantic dementia", "sickle cell disease", "large B-cell lymphoma", "auto-brewery syndrome", "bladder fermentation syndrome", "RIDDLE syndrome", "African iron overload", "Scott syndrome", "Jackson-Weiss syndrome", "Floating-Harbor syndrome", "scalp-ear-nipple syndrome", "stiff skin syndrome", "Gillespie syndrome", "C syndrome", "COACH syndrome", "Cohen syndrome", "Freeman-Sheldon syndrome", "DOORS syndrome", "Saul-Wilson syndrome", "MEND syndrome", "McLeod syndrome", "CHIME syndrome", "intestinal infectious disease", "thyroid lymphoma", "migraine with aura", "eye accommodation disease", "sleeping sickness", "dry eye syndrome", "intestinal cancer", "leprosy", "prostate cancer", "prostate carcinoma", "prostate lymphoma", "viral meningitis", "chronic meningitis", "aggressive NK-cell leukemia", "chronic leukemia", "common cold", "diaphragm disease", "beach ear", "viral pneumonia", "calcium metabolism disease", "chronic tic disorder", "glucose intolerance", "blind loop syndrome", "lymph node cancer", "Alzheimer's disease", "portal hypertension", "hypertension", "nasal cavity cancer", "malignant essential hypertension", "essential hypertension", "Edwards syndrome", "partial third-nerve palsy", "total third-nerve palsy", "benign essential hypertension", "obsessive-compulsive personality disorder", "paranoid personality disorder", "antisocial personality disorder", "tongue disease", "ovarian disease", "urinary bladder cancer", "Q fever", "spotted fever", "secondary hypertension", "respiratory failure", "pulp degeneration", "opioid abuse", "plethora of newborn", "social phobia", "cat-scratch disease", "tick paralysis", "bladder calculus", "functional diarrhea", "heart disease", "brain compression", "Frey syndrome", "blue color blindness", "heart cancer", "Lyme disease", "iron deficiency anemia", "absolute glaucoma", "protein-energy malnutrition", "bladder neck cancer", "bladder dome cancer", "bladder lymphoma", "visual epilepsy", "coronary artery anomaly", "vaginal cancer", "morbid obesity", "drug-induced mental disorder", "vaginal disease", "factor VIII deficiency", "low compliance bladder", "descending colon cancer", "cascade stomach", "paranoid schizophrenia", "cough variant asthma", "male infertility", "alcoholic hepatitis", "Graves' disease", "malaria", "complex partial epilepsy", "pathological gambling", "leukemia", "intermittent explosive disorder", "acute leukemia", "locked-in syndrome", "migraine without aura", "thyroid crisis", "autistic disorder", "Miller Fisher syndrome", "conduct disorder", "shipyard eye", "transient global amnesia", "dementia", "protein-deficiency anemia", "benign secondary hypertension", "gout", "brain cancer", "lung cancer", "Rift Valley fever", "intestinal tuberculosis", "pedophilia", "Kawasaki disease", "hairy tongue", "twin-to-twin transfusion syndrome", "frontal sinus cancer", "prostate calculus", "malignant secondary hypertension", "sweat gland disease", "sick sinus syndrome", "red color blindness", "acquired color blindness", "visual pathway disease", "facial paralysis", "bladder neck obstruction", "chronic orbital inflammation", "POEMS syndrome", "bipolar I disorder", "anus cancer", "toxic shock syndrome", "blue toe syndrome", "ovarian dysfunction", "pulp erosion", "spinal cord lymphoma", "spinal cord melanoma", "Down syndrome", "fragile X syndrome", "blindness", "generalized anxiety disorder", "mixed malaria", "Parkinson's disease", "estrogen excess", "farmer's lung", "dumping syndrome", "Foster-Kennedy syndrome", "Binder syndrome", "major depressive disorder", "Weaver syndrome", "cholera", "personality disorder", "cognitive disorder", "pulmonary systemic sclerosis", "depressive disorder", "breast cancer", "male breast cancer", "cancer", "skeletal tuberculosis", "congenital heart disease", "glaucoma", "drug psychosis", "vascular cancer", "facial nerve disease", "cardiovascular cancer", "conversion disorder", "vascular disease", "thyroid cancer", "pancreatic cancer", "generalized epilepsy", "bone cancer", "jaw cancer", "sexual dysfunction", "hepatitis C", "viral hepatitis", "melanoma", "reproductive organ cancer", "Laurence-Moon syndrome", "cerebral palsy", "anxiety disorder", "hepatitis B", "drug-induced hepatitis", "post-traumatic stress disorder", "transient arthritis", "sweat gland cancer", "heel spur", "B cell deficiency", "ovarian lymphoma", "nasal cavity disease", "ascending colon cancer", "colon cancer", "factor VII deficiency", "factor V deficiency", "factor X deficiency", "factor XI deficiency", "focal epilepsy", "motor neuron disease", "iron metabolism disease", "anemia", "pupil disease", "renal artery disease", "ovarian cancer", "protein S deficiency", "psychotic disorder", "phosphorus metabolism disease", "enamel erosion", "alcohol-induced mental disorder", "reflex epilepsy", "morphine dependence", "pancreas disease", "transverse colon cancer", "kidney cancer", "transitional cell carcinoma", "mushroom workers' lung", "sick building syndrome", "Bloom syndrome", "bone carcinoma", "tic disorder", "Dandy-Walker syndrome", "middle lobe syndrome", "nose disease", "asthma", "hairy cell leukemia", "Newcastle disease", "eye carcinoma", "motion sickness", "inner ear disease", "pulmonary tuberculosis", "renal artery obstruction", "cystic kidney disease", "carbohydrate metabolic disorder", "germ cell cancer", "substance abuse", "acute kidney failure", "food allergy", "carcinoma", "nervous system cancer", "gastrointestinal system cancer", "gastric outlet obstruction", "anus disease", "lipid metabolism disorder", "lipid pneumonia", "bipolar disorder", "mood disorder", "partial motor epilepsy", "partial sensory epilepsy", "bone inflammation disease", "accessory nerve disease", "coronary artery disease", "peripheral vascular disease", "penis carcinoma", "uterine disease", "skin carcinoma", "breast carcinoma", "iris cancer", "Turner syndrome", "chronic wasting disease", "liver cancer", "uterine cancer", "bladder disease", "Leigh disease", "wheat allergy", "olfactory nerve disease", "childhood kidney cancer", "skin disease", "nutrition disease", "protein C deficiency", "vaginal discharge", "pulmonary fibrosis", "Coffin-Lowry syndrome", "external ear disease", "skull base cancer", "Lynch syndrome", "lung carcinoma", "tuberculosis", "urinary system cancer", "bladder carcinoma", "oral tuberculosis", "mouth disease", "gastrointestinal tuberculosis", "skeletal muscle cancer", "muscle cancer", "liver disease", "tick infestation", "nail disease", "orbital cancer", "primary syphilis", "secondary syphilis", "syphilis", "autoimmune disease", "articulation disorder", "glucose metabolism disease", "brain stem cancer", "hair disease", "smooth muscle cancer", "anal margin carcinoma", "Ebola virus disease", "burning mouth syndrome", "cervical cancer", "milk allergy", "egg allergy", "peanut allergy", "nut allergy", "hepatitis E", "dyslexia", "large cell carcinoma", "bile duct cancer", "prostate disease", "mongolian spot", "space motion sickness", "movement disease", "Colorado tick fever", "middle ear carcinoma", "anal carcinoma", "bone marrow cancer", "bone marrow disease", "neurotic disorder", "essential tremor", "thyroid gland disease", "adjustment disorder", "middle ear cancer", "middle ear disease", "nutritional deficiency disease", "aortic disease", "urinary tract obstruction", "intestinal disease", "eyelid disease", "Roberts syndrome", "dental pulp disease", "sleep disorder", "schizophrenia", "stomach carcinoma", "pneumonia", "kidney disease", "Ferguson-Smith tumor", "spinal cancer", "eye disease", "large intestine cancer", "retinal disease", "Werner syndrome", "visual cortex disease", "nerve compression syndrome", "lung clear cell-sugar-tumor", "cerebral lymphoma", "gray zone lymphoma", "childhood lymphoma", "breast lymphoma", "panic disorder", "specific phobia", "animal phobia", "appendix disease", "pulmonary artery disease", "AIDS phobia", "flying phobia", "acute stress disorder", "anal canal carcinoma", "aortic valve disease", "blood protein disease", "complement deficiency", "inflammatory breast carcinoma", "migraine", "pulmonary hypertension", "inherited metabolic disorder", "muscle tissue disease", "spleen cancer", "bilateral breast cancer", "bone lymphoma", "compartment syndrome", "liver carcinoma", "skin tag", "combat disorder", "pancreatic cholera", "anthrax disease", "inhibited female orgasm", "bladder tuberculosis", "stomach disease", "retinal cell cancer", "retina lymphoma", "Coats disease", "delusional disorder", "renal infectious disease", "chronic kidney disease", "female breast carcinoma", "sporadic breast cancer", "septic arthritis", "auditory system cancer", "intestinal obstruction", "influenza", "arthritis", "hereditary night blindness", "night blindness", "lung disease", "collagen disease", "lip cancer", "soft palate cancer", "scarlet fever", "upper gum cancer", "oral cavity cancer", "measles", "substance-induced psychosis", "tongue cancer", "vein disease", "eating disorder", "genital herpes", "vascular dementia", "bacterial pneumonia", "lower lip cancer", "skin melanoma", "learning disability", "metal metabolism disorder", "liver lymphoma", "hard palate cancer", "speech disorder", "Ramsay Hunt syndrome", "mast-cell leukemia", "lip disease", "language disorder", "orbital disease", "Taylor's syndrome", "brain disease", "intrinsic asthma", "lipid storage disease", "bacterial meningitis", "meningitis", "cannabis abuse", "plasma cell leukemia", "vascular skin disease", "cervical incompetence", "yellow fever", "renal tuberculosis", "heart aneurysm", "bowel dysfunction", "eye degenerative disease", "cardiovascular syphilis", "developmental coordination disorder", "null-cell leukemia", "obesity", "substance dependence", "drug dependence", "heroin dependence", "orbit lymphoma"], "G": ["United Arab Emirates", "Dubai", "Al Muteena", "Mirdif", "Al Waheda", "Afghanistan", "Kabul", "Jabal os Saraj", "Ghazni", "Kunduz", "Albania", "Tirana", "Pogradec", "Librazhd", "Rrashbull", "Angola", "Mulenvos", "Caluquembe", "Nzagi", "Calulo", "Argentina", "Buenos Aires", "General Belgrano", "Veinticinco de Mayo", "El Palomar", "Austria", "Graz", "Innere Stadt", "Gries", "Favoriten", "Australia", "Brisbane", "Hoppers Crossing", "Umina Beach", "Bunbury", "Bosnia and Herzegovina", "Sarajevo", "Lukavac", "Prijedor", "Zenica", "Bangladesh", "Dhaka", "Khulna", "Chattogram", "Kushtia", "Belgium", "Brussels", "Kontich", "Ekeren", "Boussu", "Burkina Faso", "Ouagadougou", "Boromo", "Yako", "Bittou", "Bulgaria", "Sofia", "Rakovski", "Kyustendil", "Nova Zagora", "Burundi", "Bujumbura", "Zanandore", "Kirundo", "Vyanda", "Benin", "Cotonou", "Abomey", "Banikoara", "Adjarra", "Brunei", "Bandar Seri Begawan", "Mentiri", "Kuala Belait", "Serasa", "Bolivia", "Santa Cruz de la Sierra", "Huanuni", "Achocalla", "Sacaba", "Brazil", "Rio de Janeiro", "Sousa", "Ruy Barbosa", "Quatro Barras", "Botswana", "Gaborone", "Letlhakane", "Lobatse", "Mahalapye", "Belarus", "Minsk", "Novopolotsk", "Kalodzishchy", "Zhlobin", "Belize", "Belize City", "Orange Walk", "Belmopan", "San Ignacio", "Canada", "Toronto", "Mount Olive-Silverstone-Jamestown", "Hespeler", "McKenzie Towne", "Democratic Republic of the Congo", "Kinshasa", "Kikwit", "Katanda", "Lukula", "Central African Republic", "Bangui", "Sibut", "Paoua", "Bouar", "Republic of the Congo", "Brazzaville", "Makoua", "Loudima", "Mossendjo", "Switzerland", "Basel", "Vevey", "Luzern", "Bulle", "Ivory Coast", "Abidjan", "Aboka", "Moussadougou", "Dabou", "Chile", "Puente Alto", "Pozo Almonte", "Padre Las Casas", "Coihueco", "Cameroon", "Douala", "Nkoteng", "Kumba", "Okoa", "China", "Shanghai", "Wuzhou", "Kaitong", "Guangshun", "Colombia", "Cali", "Quimbaya", "Lorica", "Anserma", "Costa Rica", "Alajuela", "Chacarita", "Puntarenas", "San Rafael Arriba", "Cuba", "Havana", "Abreus", "Nuevitas", "Yara", "Cabo Verde", "Praia", "Espargos", "Mindelo", "Assomada", "Czechia", "Prague", "Chrudim", "Znojmo", "Trutnov", "Germany", "Berlin", "Bergisch Gladbach", "Northeim", "Seevetal", "Djibouti", "Balbala", "Arta", "Obock", "Tadjoura", "Denmark", "Copenhagen", "Kolding", "Ringsted", "Kalundborg", "Dominican Republic", "Santiago de los Caballeros", "Santo Domingo Oeste", "Neiba", "San Fernando de Monte Cristi", "Algeria", "Algiers", "Arbatache", "Souk Ahras", "Ouled Mimoun", "Ecuador", "Quito", "Riobamba", "Pelileo", "Otavalo", "Estonia", "Tallinn", "Kesklinn", "Narva", "Pirita", "Eritrea", "Asmara", "Barentu", "Mendefera", "Keren", "Spain", "Palma", "La Solana", "Leioa", "Pinto", "Ethiopia", "Addis Ababa", "Burayu", "Lasoano", "Indabaguna", "Finland", "Helsinki", "Espoonlahti", "Porvoo", "Varkaus", "Fiji", "Nasinu", "Lami", "Sigatoka", "Suva", "France", "Marseille", "Champigny-sur-Marne", "Cergy", "Maurepas", "Gabon", "Libreville", "Koulamoutou", "Port-Gentil", "Owendo", "United Kingdom", "Glasgow", "Berkhamsted", "Rawtenstall", "Newcastle under Lyme", "French Guiana", "Cayenne", "Saint-Laurent-du-Maroni", "Matoury", "Kourou", "Ghana", "Kumasi", "Nsoatre", "Wa", "Taifa", "Gambia", "Serekunda", "Latri Sabiji", "Farato", "Banjul", "Guinea", "Conakry", "Siguiri", "Banko", "Coyah", "Guadeloupe", "Les Abymes", "Sainte-Anne", "Sainte-Rose", "Capesterre-Belle-Eau", "Equatorial Guinea", "Bata", "Ebebiyin", "Ciudad de la Paz", "Malabo", "Greece", "Piraeus", "Corfu", "Rethymno", "Chios", "Guatemala", "Guatemala City", "Ciudad Vieja", "Sumpango", "Sanarate", "Guam", "Dededo Village", "Tamuning", "Yigo Village", "Tamuning-Tumon-Harmon Village", "Guinea-Bissau", "Bissau", "Xitole", "Sintra-Nema", "Contumo", "Hong Kong", "New Territories", "Braemar Hill", "Wong Tai Sin", "To Kwa Wan", "Honduras", "Tegucigalpa", "Teupasenti", "Langue", "Sulaco", "Croatia", "Zagreb", "Rijeka", "Slavonski Brod", "Split", "Haiti", "Port-au-Prince", "Grangwav", "Kenscoff", "Dessalines", "Hungary", "Budapest", "Szentendre", "Karcag", "Dabas", "Ireland", "South Dublin", "Malahide", "Sandyford", "Galway", "Israel", "Jerusalem", "Ness Ziona", "Ar Rayna", "Qiryat Ata", "India", "Mumbai", "Vettavalam", "Saoner", "Sevilimedu", "Iraq", "Baghdad", "Dihok", "Al Hillah", "Al-'Alam", "Iran", "Tehran", "Nowshahr", "Khorramshahr", "Surian", "Italy", "Milan", "Sora", "Grugliasco", "Eppan", "Jamaica", "New Kingston", "Portmore", "Spanish Town", "Montego Bay", "Jordan", "Amman", "Al Jubayhah", "Kurayyimah", "Aqaba", "Japan", "Tokyo", "Kamiiso", "Wakimachi", "Kushimoto", "Kenya", "Nairobi", "Wajir", "Takaba", "Makindu", "Kyrgyzstan", "Bishkek", "Karakol", "Kant", "Manas", "Cambodia", "Phnom Penh", "Sihanoukville", "Sisophon", "Kandoeng", "North Korea", "Pyongyang", "Chunggang", "Namyang", "Ayang-ni", "South Korea", "Seoul", "Gwangmyeong", "Ara-dong", "Seosan", "Kuwait", "Kuwait City", "Ar Riqqah", "Ad Dasmah", "Al Manqaf", "Laos", "Vientiane", "Muang Xay", "Vang Vieng", "Savannakhet", "Lebanon", "Beirut", "Baalbek", "Bsharri", "Byblos", "Sri Lanka", "Dehiwala-Mount Lavinia", "Kolonnawa", "Kurunegala", "Kataragama", "Liberia", "Gbarnga", "Foya Kamara", "Kakata", "Pleebo City", "Lesotho", "Maseru", "Butha-Buthe", "Quthing", "Teyateyaneng", "Lithuania", "Vilnius", "Alytus", "Radviliskis", "Jonava", "Latvia", "Riga", "Jelgava", "Ogre", "Salaspils", "Libya", "Benghazi", "Zintan", "Al Burayqah", "Zliten", "Morocco", "Rabat", "Ain El Aouda", "Lahraouyine", "Ben Taieb", "Moldova", "Chisinau", "Bilicenii Vechi", "Orhei", "Ungheni", "Montenegro", "Podgorica", "Pljevlja", "Bijelo Polje", "Budva", "Madagascar", "Antananarivo", "Imerimandroso", "Moramanga", "Imerintsiatosika", "North Macedonia", "Skopje", "Bogovinje", "Tearce", "Radovis", "Mali", "Bamako", "Dougabougou", "Bougouni", "Fana", "Myanmar", "Yangon", "Dawei", "Kalemyo", "Lawksawk", "Mongolia", "Ulan Bator", "Saynshand", "Bulgan", "Bogd", "Macao", "Taipa", "Zhuojiacun", "Lai Chi Van", "Luhuan", "Martinique", "Fort-de-France", "Le Robert", "Le Lamentin", "Ducos", "Mauritania", "Nouakchott", "Akjoujt", "Ayoun El Atrous", "Atar", "Malta", "Birkirkara", "Valletta", "Sliema", "Mosta", "Mauritius", "Port Louis", "Centre de Flacq", "Triolet", "Saint Pierre", "Malawi", "Lilongwe", "Nkhotakota", "Nsanje", "Mulanje", "Mexico", "Mexico City", "Tula de Allende", "Celaya", "Santiago Ixcuintla", "Malaysia", "Kuala Lumpur", "Gelang Patah", "Julau", "Sungai Udang", "Mozambique", "Maputo", "Matola", "Quelimane", "Vilankulo", "Namibia", "Windhoek", "Tsumeb", "Katutura", "Walvis Bay", "Niger", "Niamey", "Dosso", "Guidan Roumdji", "Madarounfa", "Nigeria", "Kano", "Uga", "Okrika", "Jebba", "Nicaragua", "Managua", "La Paz Centro", "Diriamba", "Bluefields", "The Netherlands", "The Hague", "Nijverdal", "Heemstede", "Amsterdam-Zuidoost", "Norway", "Oslo", "Trondheim", "Fredrikstad", "Porsgrunn", "Nepal", "Kathmandu", "Gaur", "Hetauda", "Janakpur", "New Zealand", "Auckland", "Whakatane", "Whanganui", "Manukau City", "Oman", "Muscat", "As Suwayq", "Seeb", "Bidbid", "Panama", "San Miguelito", "Parque Lefevre", "Santiago de Veraguas", "La Cabima", "Peru", "Callao", "Yurimaguas", "Atalaya", "Sechura", "Philippines", "Quezon City", "Tayabas", "Sasmuan", "Panabo", "Pakistan", "Lahore", "Ghotki", "Mananwala", "New Mirpur City", "Poland", "Warsaw", "Przasnysz", "Legionowo", "Wola", "Puerto Rico", "Ponce", "Fajardo", "Vega Baja", "Arecibo", "Palestinian Territory", "East Jerusalem", "Ramallah", "Nablus", "Battir", "Portugal", "Lisbon", "Albufeira", "Afurada de Baixo", "Amora", "Paraguay", "Ciudad del Este", "Nemby", "Presidente Franco", "Villa Elisa", "Qatar", "Doha", "Lusail", "Al Wakrah", "An Najmah", "Reunion", "Saint-Paul", "La Possession", "Le Port", "Le Tampon", "Romania", "Bucharest", "Voluntari", "Breaza", "Slatina", "Serbia", "Belgrade", "Vranje", "Loznica", "Smederevo", "Rwanda", "Kigali", "Bushenge", "Ngororero", "Nyamata", "Saudi Arabia", "Jeddah", "Baljurashi", "Turubah", "Ranyah", "Solomon Islands", "Honiara", "Nggosi", "Tandai", "Malango", "Sudan", "Khartoum", "Wad Medani", "Kuraymah", "Ghubaysh", "Sweden", "Stockholm", "Boo", "Trelleborg", "Hudiksvall", "Singapore", "Ulu Bedok", "Bukit Batok New Town", "Thomson", "Choa Chu Kang New Town", "Slovenia", "Ljubljana", "Ptuj", "Velenje", "Celje", "Slovakia", "Bratislava", "Zvolen", "Snina", "Galanta", "Sierra Leone", "Freetown", "Njala", "Koidu", "Makeni", "Senegal", "Dakar", "Tambacounda", "Matam", "Thilogne", "Somalia", "Mogadishu", "Hargeysa", "Dhamas", "Jawhar", "Suriname", "Paramaribo", "Welgelegen", "Flora", "Munder Buiten", "South Sudan", "Juba", "Yambio", "Kuacjok", "Rumbek", "El Salvador", "San Salvador", "Cojutepeque", "Quezaltepeque", "Acajutla", "Syria", "Aleppo", "An Nabk", "Manbij", "Ar Raqqah", "Chad", "N'Djamena", "Fada", "Fianga", "Bokoro", "Togo", "Kara", "Dapaong", "Tchamba", "Niamtougou", "Thailand", "Bangkok", "Lak Si", "Bang Na", "Bang Rakam", "Tajikistan", "Dushanbe", "Khorugh", "Yovon", "Danghara", "Turkmenistan", "Ashgabat", "Gyzylarbat", "Magdanly", "Seydi", "Tunisia", "Tunis", "Maatmeur", "Midoun", "Metlaoui", "Trinidad and Tobago", "Chaguanas", "Mon Repos", "Laventille", "Port of Spain", "Tanzania", "Dar es Salaam", "Makongolosi", "Babati", "Ilula", "Ukraine", "Kyiv", "Otamanivka", "Bortnychi", "Desnyanskyi", "Uganda", "Kampala", "Buyale", "Koboko", "Kireka", "United States", "New York City", "Baileys Crossroads", "East Point", "Greenburgh", "Uruguay", "Montevideo", "Las Piedras", "Colonia del Sacramento", "Durazno", "Uzbekistan", "Tashkent", "Yangiyer", "Kirguli", "Payshamba Shahri", "Venezuela", "Caracas", "Biruaca", "La Paragua", "San Juan de los Morros", "Vietnam", "Ho Chi Minh City", "Sa Dec", "Hoi An", "Hanoi", "Yemen", "Sanaa", "Ibb", "Aden", "Sa'dah", "Mayotte", "Mamoudzou", "Koungou", "Dzaoudzi", "Labattoir", "South Africa", "Johannesburg", "Verena", "Marikana", "Clydesdale", "Zambia", "Lusaka", "Isoka", "Palabana", "Sinda", "Zimbabwe", "Harare", "Mvurwi", "Chipinge", "Redcliff"]}, "corder": {"P": ["body", "head", "face", "hand", "foot", "eye", "mouth", "arm", "leg", "tooth", "bird", "fish", "horse", "tree", "flower", "car", "bicycle", "airplane", "ship", "boat", "house", "church", "computer", "watch", "piano", "gun", "shoe", "shirt", "camera", "knife"], "D": ["cardio", "resp", "gi", "nerv", "skin", "musc", "urin", "endo", "immune", "blood", "repro", "breast", "chest", "metab", "infect", "tumor", "mental", "genetic", "bact", "viral", "fungal", "parasit"], "G": ["Australia and New Zealand", "Caribbean", "Central America", "Central Asia", "Eastern Africa", "Eastern Asia", "Eastern Europe", "Melanesia", "Micronesia", "Middle Africa", "Northern Africa", "Northern America", "Northern Europe", "Polynesia", "South America", "South-eastern Asia", "Southern Africa", "Southern Asia", "Southern Europe", "Western Africa", "Western Asia", "Western Europe"]}, "md5": "f5566e013e", "n": 47170}}''')
def remake(order, corder, meta):
    """ノートの中で主張の文を作り直す（E10・E10-1 と同じ書き方）。"""
    cap = lambda s: s[:1].upper() + s[1:]
    art = lambda w: ("an " if w[:1].lower() in "aeiou" else "a ") + w
    out = []
    for x in order["P"]:
        for W in corder["P"]:
            ph = meta["wholes"][W]
            out.append(["P", x, W, f"{cap(x)} is a part of {ph}.", f"{cap(ph.split(' ', 1)[1])} is a part of {art(x)}."])
    for x in order["D"]:
        for l in corder["D"]:
            lb = meta["labels"][l]
            out.append(["D", x, l, f"{cap(x)} is {lb['art']} {lb['name']}.", None])
    for x in order["G"]:
        for g in corder["G"]:
            out.append(["G", x, g, f"{x} is located in {meta['regions'][g]}.", None])
    return out
ITEMS = {}
for tag, S in SETS.items():
    ITEMS[tag] = remake(S['order'], S['corder'], S['meta'])
    assert hashlib.md5(json.dumps(ITEMS[tag]).encode()).hexdigest()[:10] == S['md5'], ('主張の作り直しが合わない', tag)
    print(tag, len(ITEMS[tag]), '主張', {d: (len(S['order'][d]), len(S['corder'][d])) for d in 'PDG'})

## 1. 辞書を読む（E10-1 で保存したもの）と、一般的な文の資料（GenericsKB）を取って、各主張の文を探す（GPUなし、10〜20分）

In [ ]:
with gzip.open(WIKT, "rt", encoding="utf-8") as f: WKD = json.load(f)["n"]
print("辞書の見出し:", len(WKD))
from lemminflect import getAllInflections
FACTS = {tag: load_gz(f"E102_facts_{tag}") for tag in ITEMS}
if any(v is None for v in FACTS.values()):
    from datasets import load_dataset
    try:
        ds = load_dataset(GKB_ID, "generics_kb", split="train")
    except Exception as e:
        print("全部の版が読めないので best の版にします:", e); ds = load_dataset(GKB_ID, "generics_kb_best", split="train")
    cols = ds.column_names; print("列:", cols, "文の数:", len(ds))
    scol = [c for c in cols if "sentence" in c][0]
    # 列は Python のリストにしてから使う（datasets の列を1つずつ引くと1回数ミリ秒かかり、340万文で何時間にもなる。11-12 直し）
    df = ds.to_pandas(); sents = df[scol].astype(str).tolist()
    scores = df["score"].astype(float).tolist() if "score" in cols else [0.0] * len(sents); del df
    from collections import defaultdict
    TERMS = {}
    for tag, its in ITEMS.items():
        meta = SETS[tag]["meta"]
        for d, x, c, stmt, _ in its:
            for term in (x, class_term(d, c, meta)):
                if term not in TERMS: TERMS[term] = forms(term, getAllInflections)
    print("探す語:", len(TERMS))
    t = time.time(); HITS, KEPT = term_hits(sents, scores, TERMS); print(f"文を読んだ {time.time() - t:.0f}秒")
    for tag, its in ITEMS.items():
        if FACTS[tag] is not None: continue
        meta = SETS[tag]["meta"]; per, used = [], {}
        for d, x, c, stmt, _ in its:
            ids = sorted(HITS[x] & HITS[class_term(d, c, meta)])[:K_FACT]
            per.append(ids)
            for i in ids: used[i] = KEPT[i]
        FACTS[tag] = {"items": per, "sents": {str(i): s for i, s in used.items()}, "n_corpus": len(sents), "source": GKB_ID}
        save_gz(FACTS[tag], f"E102_facts_{tag}")
for tag, F_ in FACTS.items():
    per = F_["items"]
    for d in "PDG":
        m = [len(p) > 0 for p, it in zip(per, ITEMS[tag]) if it[0] == d]
        print(tag, d, "文が見つかった主張の割合", round(float(np.mean(m)), 3))

## 2. Gemmaへの問いを作る（対照の入れ替えは決まった種で）

In [ ]:
def build_prompts(tag, with_controls):
    its, meta = ITEMS[tag], SETS[tag]["meta"]; per = FACTS[tag]["items"]; S_ = FACTS[tag]["sents"]
    rng = random.Random(1)
    Q = {"def": [], "gen": []}
    if with_controls: Q["defS"] = []; Q["genS"] = []
    swap_x, swap_c, swap_f = {}, {}, {}
    if with_controls:
        for d in "PDG":
            xs = sorted({it[1] for it in its if it[0] == d}); cs = sorted({it[2] for it in its if it[0] == d})
            for L, M in ((xs, swap_x), (cs, swap_c)):
                sh = L[:]; rng.shuffle(sh)
                for a, b in zip(sh, sh[1:] + sh[:1]): M[(d, a)] = b
            withf = [i for i, it in enumerate(its) if it[0] == d and per[i]]
            sh = withf[:]; rng.shuffle(sh)
            for a, b in zip(sh, sh[1:] + sh[:1]): swap_f[a] = b
    for i, (d, x, c, stmt, _) in enumerate(its):
        w = class_term(d, c, meta)
        Q["def"].append(def_prompt(x, defs_text(WKD, x), w, defs_text(WKD, w), stmt))
        Q["gen"].append(fact_prompt([S_[str(j)] for j in per[i]], stmt))
        if with_controls:
            x2, c2 = swap_x[(d, x)], swap_c[(d, c)]; w2 = class_term(d, c2, meta)
            Q["defS"].append(def_prompt(x2, defs_text(WKD, x2), w2, defs_text(WKD, w2), stmt))
            j = swap_f.get(i)
            Q["genS"].append(fact_prompt([S_[str(k)] for k in per[j]] if j is not None else [], stmt))
    return Q
PROMPTS = {"N": build_prompts("N", True), "O": build_prompts("O", False)}
print({t: {k: len(v) for k, v in q.items()} for t, q in PROMPTS.items()})
print(PROMPTS["N"]["def"][0]); print(); print(PROMPTS["N"]["gen"][0])

## 3. Gemmaに聞く（GPU、60〜90分。方法ごとにドライブに保存）

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
tok = AutoTokenizer.from_pretrained(GEMMA_ID, token=userdata.get("HF_TOKEN"), padding_side="left")
gm = AutoModelForCausalLM.from_pretrained(GEMMA_ID, token=userdata.get("HF_TOKEN"), torch_dtype=torch.float16, device_map="cuda").eval()
YES = tok.encode("Yes", add_special_tokens=False)[0]; NO = tok.encode("No", add_special_tokens=False)[0]
def yes_no(qs, B=32, log_every=200):
    # 各問いの最後の位置で、Yes と No の値の差だけを残す（全部の語の値をGPUに溜めない）
    out = []; t = time.time()
    for i in range(0, len(qs), B):
        ps = [tok.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True) for q in qs[i:i + B]]
        enc = tok(ps, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        with torch.no_grad():
            r = gm(**enc, logits_to_keep=1).logits[:, -1, :].float()
            out.extend((r[:, YES] - r[:, NO]).cpu().numpy().round(3).tolist()); del r
        if (i // B) % log_every == 0: print(f"  {i}/{len(qs)}  {time.time() - t:.0f}秒")
    return out
for tag, Q in PROMPTS.items():
    for m, qs in Q.items():
        name = f"E102_gemma_{tag}_{m}"
        if load_gz(name) is not None: print("済み:", name); continue
        print(name, len(qs)); save_gz({"logit": yes_no(qs), "model": GEMMA_ID}, name)
# 確かめ：説明文なしの聞き方を、E10-1 の問題の最初の200個だけもう一度（E10-1 の値と合うか）
if load_gz("E102_check_plain") is None:
    chk = yes_no([gemma_q(it[3]) for it in ITEMS["N"][:200]])
    save_gz({"logit": chk}, "E102_check_plain")
del gm; torch.cuda.empty_cache()

## 4. 一般的な文をNLIにかける（⑪n、GPU、10〜20分）

In [ ]:
ntok = AutoTokenizer.from_pretrained(NLI_ID)
nm = AutoModelForSequenceClassification.from_pretrained(NLI_ID).cuda().eval()
lab = {v.lower(): int(k) for k, v in nm.config.id2label.items()}
ORD = [lab["entailment"], lab["contradiction"]]
def nli(P, H, B=128):
    out = np.zeros((len(P), 2), np.float32)
    for i in range(0, len(P), B):
        enc = ntok(P[i:i + B], H[i:i + B], return_tensors="pt", padding=True, truncation=True, max_length=192).to("cuda")
        with torch.no_grad():
            pr = torch.softmax(nm(**enc).logits.float(), -1)[:, ORD]
        out[i:i + len(pr)] = pr.cpu().numpy()
    return out
for tag in ITEMS:
    name = f"E102_nli_{tag}"
    if load_gz(name) is not None: print("済み:", name); continue
    per = FACTS[tag]["items"]; S_ = FACTS[tag]["sents"]
    P, H, owner = [], [], []
    for i, ids in enumerate(per):
        for j in ids: P.append(S_[str(j)]); H.append(ITEMS[tag][i][3]); owner.append(i)
    pr = nli(P, H) if P else np.zeros((0, 2))
    ent = [None] * len(per); con = [None] * len(per)
    for (e, c), i in zip(pr.tolist(), owner):
        if ent[i] is None or e > ent[i]: ent[i], con[i] = round(e, 4), round(c, 4)
    save_gz({"ent": ent, "con": con, "n_pairs": len(P), "model": NLI_ID}, name)

## 5. できたファイル

In [ ]:
for f in sorted(os.listdir(OUT)):
    print(f, round(os.path.getsize(f"{OUT}/{f}") / 1e6, 2), "MB")